# SHAP = SHapley Additive exPlanations

Aula: explicar as decisões de um modelo que não se explica sozinho (`LinearSVC`) usando SHAP.

In [ ]:
import shap
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC

# o KernelExplainer sorteia combinações de variáveis; fixar a seed deixa o resultado igual em toda execução
np.random.seed(42)

In [ ]:
# faltava carregar o dataset
data = load_breast_cancer()
X, y = data['data'], data['target']

In [ ]:
# 0 = malignant, 1 = benign
data['target_names']

In [ ]:
# random_state fixo pra dar o mesmo resultado em toda execução
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
clf = LinearSVC(max_iter=20000, random_state=42)
clf.fit(X_train, y_train)

In [ ]:
clf.score(X_test, y_test)

In [ ]:
# só devolve 0 ou 1
clf.predict([X_test[0]])

In [ ]:
# KernelExplainer funciona com qualquer modelo
# o kmeans resume o treino em 10 pontos pra ficar mais rápido
explainer = shap.KernelExplainer(clf.predict, shap.kmeans(X_train, 10))

In [ ]:
# calcula o SHAP de cada exemplo do teste
shap_values = explainer.shap_values(X_test)

In [ ]:
# explicação individual do primeiro exemplo do teste
shap.initjs()
shap.force_plot(explainer.expected_value, shap_values[0], X_test[0], feature_names=data['feature_names'])

In [ ]:
# importância global = média do valor absoluto do SHAP de cada variável
mean_abs_shap_values = np.abs(shap_values).mean(axis=0)

In [ ]:
feature_names = data['feature_names']
feature_importance = zip(feature_names, mean_abs_shap_values)

In [ ]:
sorted_feature_importances = sorted(feature_importance, key=lambda x: x[1], reverse=True)

In [ ]:
for feature, importance in sorted_feature_importances:
    print(feature, ':', importance)